In [1]:
import polars as pl
from datetime import datetime
import numpy as np

In [ ]:
ENDERECO_DADOS = "../../DADOS/auxiliobrasil.parquet"

pl.Config.set_float_precision(2)
pl.Config.set_thousands_separator(".")
pl.Config.set_tbl_rows(-1)


try:
    print("Obtendo os dados...")
    inicio = datetime.now()

    df_lazy_plan = (
        pl.scan_parquet(ENDERECO_DADOS)
        .select(["NOME MUNICÍPIO", "VALOR PARCELA"])
        .with_columns(
            pl.col("NOME MUNICÍPIO").cast(pl.Categorical)
        )
        .filter(pl.col("VALOR PARCELA") > 2000)
        .group_by('NOME MUNICÍPIO')
        .agg(pl.col("VALOR PARCELA").sum())
        .sort("VALOR PARCELA", descending=True)
    )
    
    df_auxilio = df_lazy_plan.collect()

    print(df_auxilio.head())
    fim = datetime.now()
    print(f"Tempo Decorrido: {fim - inicio}")

except Exception as e:
    print(f"Erro ao obter dados {e}")

In [5]:
try:
    array = np.array(df_auxilio["VALOR PARCELA"])
    media = np.mean(array)
    mediana = np.median(array)
    distancia_percentual = abs((media - mediana) / mediana) * 100

    print(f"Media: {media:.2f}")
    print(f"Mediana: {mediana:.2f}")
    print(f"Distancia: {distancia_percentual:.2f}%")


except Exception as e:
    print(f"Erro ao obter medidas estatísticas {e}")

Media: 17916.60
Mediana: 8372.50
Distancia: 113.99%


## Quartis

In [6]:
# calculando os quartis e o IQR
q1 = np.percentile(array, 25)
q3 = np.percentile(array, 75)
iqr = q3 - q1
print(f"Q1: {q1:.2f}")
print(f"Q3: {q3:.2f}")
print(f"IQR: {iqr:.2f}")

Q1: 2877.00
Q3: 26253.00
IQR: 23376.00
